# 誤植なし正例 30件テスト

* `doi_list.txt` の先頭 30 件を対象にする
* MongoDB から DOI 指定で取得する
* 3 スタイルへ整形し、テスト用ファイルとして出力する
* `find_one` にかかった時間を各 DOI ごとに表示する

In [1]:
from pathlib import Path
import time
import re
import pymongo

url_mongodb = "mongodb://localhost:27017/"
myclient = pymongo.MongoClient(url_mongodb)
mydb = myclient["jalc"]
mycol = mydb["restapi"]

doi_list_path = Path("../5データ/5-2評価実験用データ/5-2-1参考文献文字列_誤植なし（実在）/doi_list.txt")
output_dir = Path("../5データ/5-2評価実験用データ/5-2-1参考文献文字列_誤植なし（実在）")

scj_registered_AcademicSocieties = []
with open("../5データ/5-3日本学術会議協力学術研究団体/scj_registered_AcademicSocieties.txt", "r") as f:
    lines = f.readlines()
for line in lines:
    scj_registered_AcademicSocieties += line.split(",")

In [2]:
def has_japanese(text):
    return bool(re.search(r"[\u3040-\u309F\u30A0-\u30FF\uFF66-\uFF9F\u4E00-\u9FFF]", text))

def check_lang(doc):
    title_lang, creator_lang, journal_lang = "", "", ""
    for title in doc["title_list"]:
        lang = title.get("lang", "")
        if lang == "" and has_japanese(title.get("title", "")):
            lang = "ja"
        if lang == "ja":
            title_lang = "ja"
            break

    for creator in doc["creator_list"]:
        for creator_name in creator["names"]:
            lang = creator_name.get("lang", "")
            if lang == "" and has_japanese(creator_name.get("first_name", "")):
                lang = "ja"
            if lang == "ja":
                creator_lang = "ja"
                break

    for journal in doc["journal_title_name_list"]:
        lang = journal.get("lang", "")
        if lang == "" and has_japanese(journal.get("journal_title_name", "")):
            lang = "ja"
        if lang == "ja":
            journal_lang = "ja"
            break

    return title_lang == "ja" and creator_lang == "ja" and journal_lang == "ja"

def check_publisher(doc, academic_societies):
    for society in academic_societies:
        for publisher in doc["publisher_list"]:
            if society in publisher.get("publisher_name", ""):
                return True
    return False

def get_title(doc):
    title_text = ""
    for val in doc["title_list"]:
        lang = val.get("lang", "")
        title = val.get("title", "")
        subtitle = val.get("subtitle", "")
        if lang == "" and has_japanese(title):
            lang = "ja"
        if lang == "ja":
            title_text = title
            if subtitle != "":
                title_text = title_text + " " + subtitle
    return title_text

def get_journal(doc):
    journal_text = ""
    for val in doc["journal_title_name_list"]:
        journal_title = val.get("journal_title_name", "")
        kind = val.get("type", "")
        lang = val.get("lang", "")
        if lang == "" and has_japanese(journal_title):
            lang = "ja"
        if lang == "ja":
            journal_text = journal_title
            if kind == "full":
                break
    return journal_text

def get_creator(doc):
    creator_list = []
    for val in doc["creator_list"]:
        if len(val["names"]) == 1:
            last_name = val["names"][0].get("last_name", "")
            first_name = val["names"][0].get("first_name", "")
            creator_list.append([last_name, first_name])
        else:
            for name in val["names"]:
                last_name = name.get("last_name", "")
                first_name = name.get("first_name", "")
                lang = name.get("lang", "")
                if has_japanese(first_name):
                    lang = "ja"
                if lang == "ja":
                    creator_list.append([last_name, first_name])
                    break
    return creator_list

def create_reference_jsai(creator_list, title, journal, volume, issue, page_list, year):
    tmp = []
    for name in creator_list:
        if name[0] != "":
            if has_japanese(name[0]):
                tmp.append(name[0])
            else:
                tmp.append(name[0] + ", " + name[1][0] + ".")
        else:
            tmp.append(name[1])
    creator_text = "，".join(tmp)
    return creator_text + "：" + title + "，" + journal + "，" + "Vol." + volume + ", " + "No." + issue + ", " + "pp." + page_list[0] + "-" + page_list[1] + " (" + year + ")."

def create_reference_ipsj(creator_list, title, journal, volume, issue, page_list, year):
    creator_text = ""
    tmp = []
    num = 0
    for name in creator_list:
        num += 1
        if num == 4:
            creator_text = "ほか"
            break
        if name[0] != "":
            if has_japanese(name[0]):
                tmp.append(name[0] + name[1])
            else:
                tmp.append(name[0] + ", " + name[1][0] + ".")
        else:
            tmp.append(name[1])
    creator_text = "，".join(tmp) + creator_text
    return creator_text + "：" + title + "，" + journal + "，" + "Vol." + volume + "，" + "No." + issue + "，" + "pp." + page_list[0] + "-" + page_list[1] + "（" + year + "）．"

def create_reference_lsj(creator_list, title, journal, volume, issue, page_list, year):
    tmp = []
    for name in creator_list:
        tmp.append(name[0] + name[1])
    return "・".join(tmp) + "（" + year + "）" + "「" + title + "」" + "『" + journal + "』" + volume + "（" + issue + "）: " + page_list[0] + "–" + page_list[1] + "."

In [3]:
with open(doi_list_path, "r") as f:
    doi_list = [line.strip() for line in f.readlines() if line.strip()]

target_doi_list = doi_list[:30]
print(f"target_doi_num: {len(target_doi_list)}")

reference_jsai_list = []
reference_ipsj_list = []
reference_lsj_list = []
elapsed_list = []

for idx, target_doi in enumerate(target_doi_list, start=1):
    start = time.perf_counter()
    doc = mycol.find_one({"doi": target_doi}, {"_id": 0})
    elapsed = time.perf_counter() - start
    elapsed_list.append(elapsed)
    print(f"{idx:02}: {target_doi} | mongo_find_one_sec: {elapsed:.6f}")

    if doc is None:
        raise ValueError(f"MongoDB に DOI が見つかりません: {target_doi}")
    if not check_lang(doc):
        raise ValueError(f"日本語条件を満たしません: {target_doi}")
    if not check_publisher(doc, scj_registered_AcademicSocieties):
        raise ValueError(f"学術会議登録団体条件を満たしません: {target_doi}")

    title = get_title(doc)
    journal = get_journal(doc)
    volume = doc["volume"]
    issue = doc["issue"]
    page_list = [doc["first_page"], doc["last_page"]]
    year = doc["publication_date"]["publication_year"]
    creator_list = get_creator(doc)

    reference_jsai_list.append(create_reference_jsai(creator_list.copy(), title, journal, volume, issue, page_list.copy(), year))
    reference_ipsj_list.append(create_reference_ipsj(creator_list.copy(), title, journal, volume, issue, page_list.copy(), year))
    reference_lsj_list.append(create_reference_lsj(creator_list.copy(), title, journal, volume, issue, page_list.copy(), year))

outputs = {
    output_dir / "positive_reference_jsai_eval_none_typo_test_30.txt": "\n".join(reference_jsai_list),
    output_dir / "positive_reference_ipsj_eval_none_typo_test_30.txt": "\n".join(reference_ipsj_list),
    output_dir / "positive_reference_lsj_eval_none_typo_test_30.txt": "\n".join(reference_lsj_list),
    output_dir / "doi_list_test_30.txt": "\n".join(target_doi_list),
}

for path, text in outputs.items():
    with open(path, "w") as f:
        f.write(text)
    print(f"written: {path}")

print(f"mongo_find_one_total_sec: {sum(elapsed_list):.6f}")
print(f"mongo_find_one_avg_sec: {sum(elapsed_list) / len(elapsed_list):.6f}")

print("--- JSAI first ---")
print(reference_jsai_list[0])
print("--- IPSJ first ---")
print(reference_ipsj_list[0])
print("--- LSJ first ---")
print(reference_lsj_list[0])

target_doi_num: 30
01: 10.2750/jrps.2.1_1 | mongo_find_one_sec: 0.005631
02: 10.2750/jrps.2.1_17 | mongo_find_one_sec: 0.001170
03: 10.2750/jrps.2.1_27 | mongo_find_one_sec: 0.000585
04: 10.2750/jrps.2.1_33 | mongo_find_one_sec: 0.000351
05: 10.2750/jrps.2.1_44 | mongo_find_one_sec: 0.000339
06: 10.2750/jrps.2.1_54 | mongo_find_one_sec: 0.000281
07: 10.2750/jrps.2.1_62 | mongo_find_one_sec: 0.000233
08: 10.2750/jrps.2.1_69 | mongo_find_one_sec: 0.000436
09: 10.2750/jrps.2.1_7 | mongo_find_one_sec: 0.000434
10: 10.2750/jrps.2.1_77 | mongo_find_one_sec: 0.000253
11: 10.2750/jrps.2.1_87 | mongo_find_one_sec: 0.000268
12: 10.7222/marketingreview.2022.001 | mongo_find_one_sec: 0.001790
13: 10.7222/marketingreview.2022.002 | mongo_find_one_sec: 0.000830
14: 10.7222/marketingreview.2022.003 | mongo_find_one_sec: 0.000468
15: 10.7222/marketingreview.2022.004 | mongo_find_one_sec: 0.000571
16: 10.7222/marketingreview.2022.005 | mongo_find_one_sec: 0.000349
17: 10.7222/marketingreview.2022.006 |